In [1]:
import joblib
import json

loaded_model = joblib.load('churn_model_xgb.pkl')
loaded_scaler = joblib.load('churn_scaler.pkl')

with open('model_config.json', 'r') as f:
    config = json.load(f)

print(config)

{'threshold': 0.21, 'model': 'XGBoost', 'features': ['account length', 'area code', 'international plan', 'voice mail plan', 'total day calls', 'total day charge', 'total eve calls', 'total eve charge', 'total night calls', 'total night charge', 'total intl calls', 'total intl charge', 'customer service calls']}


In [3]:
import pandas as pd

new_customer = {
    'account length': 100,
    'area code': 415,
    'international plan': 1,
    'voice mail plan': 0,
    'total day calls': 110,
    'total day charge': 45.0,
    'total eve calls': 100,
    'total eve charge': 16.5,
    'total night calls': 100,
    'total night charge': 9.0,
    'total intl calls': 4,
    'total intl charge': 2.7,
    'customer service calls': 5
}
new_customer_df = pd.DataFrame([new_customer])
print(new_customer_df.shape)
print(new_customer_df)

(1, 13)
   account length  area code  international plan  voice mail plan  \
0             100        415                   1                0   

   total day calls  total day charge  total eve calls  total eve charge  \
0              110              45.0              100              16.5   

   total night calls  total night charge  total intl calls  total intl charge  \
0                100                 9.0                 4                2.7   

   customer service calls  
0                       5  


In [5]:
new_customer_scaled = loaded_scaler.transform(new_customer_df)

churn_probability = loaded_model.predict_proba(new_customer_scaled)[:, 1]
print("Churn probability:", churn_probability)

threshold = config['threshold']
prediction = (churn_probability >= threshold).astype(int)
print("Prediction (1=churn, 0=no churn):", prediction)

Churn probability: [0.8365905]
Prediction (1=churn, 0=no churn): [1]


In [7]:
def predict_churn(customer_dict):
    new_df = pd.DataFrame([customer_dict])
    scaled = loaded_scaler.transform(new_df)
    probability = loaded_model.predict_proba(scaled)[:, 1]
    threshold = config['threshold']
    prediction = (probability >= threshold).astype(int)
    return probability[0], prediction[0]

In [8]:
prob,pred = predict_churn(new_customer)
print(f"churn probability: {prob:.3f}, prediction: {pred}")

churn probability: 0.837, prediction: 1
